# Gauge-origin independence for London kinetic integrals

In [1]:
# Force the local gqcpy to be imported.
import sys
sys.path.insert(0, '../../build/gqcpy/')

import gqcpy
import numpy as np

np.set_printoptions(precision=8, linewidth=120)

## Set up molecules

In [2]:
H_atom = gqcpy.Molecule([gqcpy.Nucleus(1, 0.0, 0.0, 0.0)])
H_atom_moved = gqcpy.Molecule([gqcpy.Nucleus(1, 0.0, 1.0, 0.0)])
H_2 = gqcpy.Molecule([gqcpy.Nucleus(1, 0.0, 0.0, 0.0), gqcpy.Nucleus(1, 0.0, 1.0, 0.0)])

## Set up bases

In [3]:
# molecule = H_atom
molecule = H_atom_moved
# molecule = H_2

print(molecule)

Number of electrons: 1 
H  (0, 1, 0)



In [4]:
B0 = gqcpy.HomogeneousMagneticField([0.0, 0.0, 0], [0.0, 0.0, 0.0])
B1 = gqcpy.HomogeneousMagneticField([0.0, 0.0, -1], [0.0, 0.0, 0.0])
B1_randomG = gqcpy.HomogeneousMagneticField([0.0, 0.0, -1], [124.21, 6.34, 0.564])

In [5]:
basisset = 'STO-3G'

spinor_basis_noB = gqcpy.RSpinOrbitalBasis_d(molecule, basisset)
spinor_basis_B0 = gqcpy.LondonRSpinOrbitalBasis(molecule, basisset, B0)
spinor_basis_B1 = gqcpy.LondonRSpinOrbitalBasis(molecule, basisset, B1)
spinor_basis_B1_randomG = gqcpy.LondonRSpinOrbitalBasis(molecule, basisset, B1_randomG)

## Evaluate AO at point in space

We can also evaluate London AOs at a specific point in space.

In [25]:
eval_loc = np.array([-1, 0, 0])

print(spinor_basis_noB.evalBasisSetAtPoint(eval_loc))
print(spinor_basis_B0.evalBasisSetAtPoint(eval_loc))
print(spinor_basis_B1.evalBasisSetAtPoint(eval_loc))
print(spinor_basis_B1_randomG.evalBasisSetAtPoint(eval_loc))

[0.1367475106356501]
[(0.1367475106356501+0j)]
[(0.1200072307157648+0.06556024893928053j)]
[(-0.1218209546423018-0.06212516941695598j)]


In [26]:
spinor_basis_B1.evalBasisSetAtPoint(eval_loc)

[(0.1200072307157648+0.06556024893928053j)]

In [27]:
def get_phase_factor_from_value(val):
    # since LAO = GTO(cos(\phi) - i sin(\phi)) with \phi the phase argument (phase factor exp(-i \phi))
    return np.arctan(-val.imag/val.real)

In [ ]:
get_phase_factor_from_value(spinor_basis_B1.evalBasisSetAtPoint(eval_loc)[0]) # k \cdot r

-0.5000000000000001

In [29]:
def numerical_gradient_basis(
    basis,
    r,
    h=1e-5,
):
    r = np.asarray(r, dtype=float)
    f0 = np.asarray(basis.evalBasisSetAtPoint(r), dtype=complex)

    n_basis = len(f0)
    grad = np.zeros((n_basis, 3), dtype=complex)

    for j in range(3):
        dr = np.zeros(3)
        dr[j] = h

        f_plus  = np.asarray(basis.evalBasisSetAtPoint(r + dr), dtype=complex)
        f_minus = np.asarray(basis.evalBasisSetAtPoint(r - dr), dtype=complex)

        grad[:, j] = (f_plus - f_minus) / (2.0 * h)

    return grad

In [32]:
print('numerical:', numerical_gradient_basis(spinor_basis_B1, eval_loc))
print('analytic function:', spinor_basis_B1.evalGradBasisSetAtPoint(eval_loc))

numerical: [[0.13640738-0.00339179j 0.10362725+0.05661183j 0.        +0.j        ]]
analytic function: [[(0.13640737715649182-0.003391789199383762j), (0.10362725268685155+0.05661182615849864j), 0j]]


In [37]:
# since \grad LAO = exp(-i k \cdot r) (\grad GTO - i k GTO)
# \grad LAO / LAO = \grad GTO / GTO - i k
# so need to check if imag part of \grad LAO / LAO = k

# should give k = (-1/2, 0, 0) for G = (0,0,0), H at (0, 1, 0) and B = (0, 0, 1)

# x comp
print('x comp:', (spinor_basis_B1.evalGradBasisSetAtPoint(eval_loc)[0][0]/spinor_basis_B1.evalBasisSetAtPoint(eval_loc)[0]).imag)
# y comp
print('y comp:', (spinor_basis_B1.evalGradBasisSetAtPoint(eval_loc)[0][1]/spinor_basis_B1.evalBasisSetAtPoint(eval_loc)[0]).imag)
# z comp
print('z comp:', (spinor_basis_B1.evalGradBasisSetAtPoint(eval_loc)[0][2]/spinor_basis_B1.evalBasisSetAtPoint(eval_loc)[0]).imag)

x comp: -0.5000000000000001
y comp: 0.0
z comp: 0.0


## Adding phase factor manually on GTO

In [39]:
def compute_phase_factor(r, R_K=np.array([0, 0, 0]), B=np.array([0, 0, 0]), G=np.array([0, 0, 0])):
    k = 1/2 * (np.cross(B, R_K - G))
    return np.exp(-1.0j*np.dot(k, r))

In [44]:
phase_factor = compute_phase_factor(eval_loc, R_K=np.array([0, 1, 0]), G=np.array([0, 0, 0]), B=np.array([0, 0, -1]))
gto_r = spinor_basis_noB.evalBasisSetAtPoint(eval_loc)[0]
manual_lao_r = phase_factor * gto_r
print('manual:', manual_lao_r)
print('gqcp:', spinor_basis_B1.evalBasisSetAtPoint(eval_loc)[0])

manual: (0.12000723071576481+0.06556024893928053j)
gqcp: (0.1200072307157648+0.06556024893928053j)


In [ ]:
# compute gradient
